In [25]:
import pandas as pd

In [3]:
from pathlib import Path

In [4]:
data_path = Path(r"C:\Users\sungu\OneDrive\Desktop\Data analyst\capstone\eCommerce")

In [5]:
csv_files = {
    "2019-10": "2019-Oct.csv", 
    "2019-11": "2019-Nov.csv", 
    "2019-12": "2019-Dec.csv", 
    "2020-01": "2020-Jan.csv", 
    "2020-02": "2020-Feb.csv"
}

dtype_map = {
    "event_type": "category",
    "product_id": "int64",
    "category_id": "int64",
    "category_code": "string",
    "brand": "string",
    "price": "float64",
    "user_id": "int64", 
}

files_path = {}
for key, name in csv_files.items():
    files_path[key] = data_path / name

dfs = []
for key, fp in files_path.items():
    df = pd.read_csv(fp, dtype=dtype_map, parse_dates=["event_time"])
    df["year_month"] = key
    dfs.append(df)

df_raw = pd.concat(dfs, ignore_index=True)
print(f"合并后总行数：{df_raw.shape[0]:,}")

合并后总行数：20,692,840


In [ ]:
df_raw.to_csv(r"C:\Users\sungu\OneDrive\Desktop\Data analyst\capstone\eCommerce\merged_all.csv", index=False)

In [26]:
df=pd.read_csv("merged_all.csv")

In [27]:
df.columns

Index(['event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session',
       'year_month'],
      dtype='object')

In [13]:
df.isna().sum()

event_time              0
event_type              0
product_id              0
category_id             0
category_code    20339246
brand             8757117
price                   0
user_id                 0
user_session         4598
year_month              0
dtype: int64

In [ ]:
df.describe()

In [ ]:
df = df.dropna(subset=['user_session'])
print(f"After dropping null sessions: {df.shape}")

After dropping null sessions: (20688242, 10)


In [29]:
df = df.drop_duplicates()
print(f"After dropping duplicates: {df.shape}")

After dropping duplicates: (19579657, 10)


In [30]:
df['category_code'] = df['category_code'].fillna('unknown')
df['brand'] = df['brand'].fillna('unknown')

In [ ]:
# Investigate price distribution 

print("Price Summary")
print(df['price'].describe())

print("\nZero prices")
print(f"Count: {(df['price'] == 0).sum()}")
print(df[df['price'] == 0]['event_type'].value_counts())

print("\nNegative prices")
neg = df[df['price'] < 0]
print(f"Count: {len(neg)}")
print(neg['event_type'].value_counts())
print(neg[['event_time','event_type','product_id','price','user_id','user_session']].head(20))

print("\nNull prices")
print(f"Count: {df['price'].isna().sum()}")

Price Summary
count    1.957965e+07
mean     8.740068e+00
std      1.976439e+01
min     -7.937000e+01
25%      2.100000e+00
50%      4.110000e+00
75%      7.140000e+00
max      3.277800e+02
Name: price, dtype: float64

Zero prices
Count: 86325
event_type
cart                37357
view                30489
remove_from_cart    18478
purchase                1
Name: count, dtype: int64

Negative prices
Count: 119
event_type
purchase    119
Name: count, dtype: int64
                        event_time event_type  product_id  price    user_id  \
112860   2019-10-01 19:10:56+00:00   purchase     5716857 -23.81  552507528   
198302   2019-10-02 08:30:03+00:00   purchase     5716855  -7.94  550375225   
436918   2019-10-03 17:37:04+00:00   purchase     5716859 -47.62  555414763   
443204   2019-10-03 18:25:39+00:00   purchase     5670257 -15.87  556383221   
1295556  2019-10-09 14:49:14+00:00   purchase     5716855  -7.94  514562574   
1426186  2019-10-10 14:33:29+00:00   purchase     5716855  -

In [ ]:
# Flag returns (keep them, they're real business signals) 
df['is_return'] = (df['price'] < 0) & (df['event_type'] == 'purchase')
print(df[df['is_return']]['price'].describe())

count    119.000000
mean     -30.879328
std       20.505334
min      -79.370000
25%      -47.620000
50%      -23.810000
75%      -15.870000
max       -7.940000
Name: price, dtype: float64


In [37]:
# Drop negative price on non-purchase events
bad = (df['price'] < 0) & (df['event_type'] != 'purchase')
df = df.drop(index=df[bad].index)
print(f"Dropped: {bad.sum()} rows")

Dropped: 5 rows


In [ ]:
# drop zero-price purchase
zero_purchase = (df['price'] == 0) & (df['event_type'] == 'purchase')
print(f"Zero-price purchases: {zero_purchase.sum()}")

df = df.drop(index=df[zero_purchase].index)
print(f"Dropped: {zero_purchase.sum()} rows")
print(f"Remaining: {len(df)}")

Zero-price purchases: 1
Dropped: 1 rows
Remaining: 19579651


In [45]:
upper = df['price'].quantile(0.999)
print(f"99.9th percentile: ${upper}")
print(f"Max price: ${df['price'].max()}")
print(f"Rows above 99.9th: {(df['price'] > upper).sum()}")

99.9th percentile: $236.51
Max price: $327.78
Rows above 99.9th: 18363


In [43]:
outliers = df[df['price'] > upper]
print(outliers['event_type'].value_counts())
print(outliers['price'].describe())

event_type
view                16201
cart                 1312
remove_from_cart      727
purchase              123
Name: count, dtype: int64
count    18363.000000
mean       292.280033
std         17.035016
min        252.380000
25%        273.020000
50%        299.810000
75%        299.810000
max        327.780000
Name: price, dtype: float64


In [47]:
print(df['category_code'].value_counts())

category_code
unknown                                   19236765
appliances.environment.vacuum               145793
stationery.cartrige                          56761
apparel.glove                                49823
furniture.living_room.cabinet                30120
accessories.bag                              24084
furniture.bathroom.bath                      23158
appliances.personal.hair_cutter               5324
accessories.cosmetic_bag                      3541
appliances.personal.massager                  3249
appliances.environment.air_conditioner         691
furniture.living_room.chair                    338
sport.diving                                     4
Name: count, dtype: int64


In [48]:
print(df['category_id'].nunique())
print(df['brand'].value_counts().head(20))

525
brand
unknown      8263663
runail       1433792
irisk         970331
grattol       811021
masura        801532
bpw.style     409838
ingarden      402728
estel         348312
kapous        314395
jessnail      245304
uno           239727
freedecor     222409
italwax       216991
concept       195157
bluesky       189424
pole          185286
haruyama      178415
cnd           171653
milv          156581
domix         150776
Name: count, dtype: int64


In [ ]:
# will use category_id for category-level analysis
print(df.groupby('category_id')['event_type'].value_counts().unstack().head(10))

event_type              cart  purchase  remove_from_cart     view
category_id                                                      
1487580004807082827     11.0       4.0               3.0     93.0
1487580004832248652  19273.0    2957.0           11980.0  30514.0
1487580004857414477  23288.0    3854.0           17487.0  31766.0
1487580004882580302  16607.0    3157.0           10032.0  23049.0
1487580004916134735  74229.0   15112.0           48681.0  89693.0
1487580004941300560     78.0      10.0              38.0    454.0
1487580004966466385      3.0       NaN               3.0     35.0
1487580004983243602    141.0       6.0              39.0    457.0
1487580005008409427  20142.0    3114.0           13248.0  26552.0
1487580005025186644    195.0      16.0              88.0   3044.0


In [50]:
print(df["event_time"].head(2))

0    2019-10-01 00:00:00+00:00
1    2019-10-01 00:00:03+00:00
Name: event_time, dtype: object


In [ ]:
#Timezone Standardization
df['event_time'] = pd.to_datetime(df['event_time'], utc=True).dt.tz_convert('Europe/Moscow')

**Initial Feature Engineering**

In [ ]:
# extract time features
df['date'] = df['event_time'].dt.date
df['hour'] = df['event_time'].dt.hour
df['day_of_week'] = df['event_time'].dt.day_name()

print(df[['event_time','date','hour','day_of_week']].head())

                 event_time        date  hour day_of_week
0 2019-10-01 03:00:00+03:00  2019-10-01     3     Tuesday
1 2019-10-01 03:00:03+03:00  2019-10-01     3     Tuesday
2 2019-10-01 03:00:07+03:00  2019-10-01     3     Tuesday
3 2019-10-01 03:00:07+03:00  2019-10-01     3     Tuesday
4 2019-10-01 03:00:15+03:00  2019-10-01     3     Tuesday


In [ ]:
print(df['hour'].value_counts().sort_values())
#a pattern consistent with expected Russian consumer behavior,the conversion to Moscow timezone was validated 


hour
4      173120
5      189854
3      191668
6      263912
2      276307
7      374364
1      477660
8      568456
9      770340
0      797691
10     919203
18    1007573
11    1012906
19    1043480
17    1049949
12    1074086
20    1116955
16    1118268
13    1130669
23    1159123
14    1167735
15    1181174
21    1227898
22    1287260
Name: count, dtype: int64


In [61]:
df.head(5)

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,year_month,is_return,date,hour,day_of_week
0,2019-10-01 03:00:00+03:00,cart,5773203,1487580005134238553,unknown,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885,2019-10,False,2019-10-01,3,Tuesday
1,2019-10-01 03:00:03+03:00,cart,5773353,1487580005134238553,unknown,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885,2019-10,False,2019-10-01,3,Tuesday
2,2019-10-01 03:00:07+03:00,cart,5881589,2151191071051219817,unknown,lovely,13.48,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9,2019-10,False,2019-10-01,3,Tuesday
3,2019-10-01 03:00:07+03:00,cart,5723490,1487580005134238553,unknown,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885,2019-10,False,2019-10-01,3,Tuesday
4,2019-10-01 03:00:15+03:00,cart,5881449,1487580013522845895,unknown,lovely,0.56,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9,2019-10,False,2019-10-01,3,Tuesday


In [ ]:
# Session-level aggregation
event_dummies = pd.get_dummies(df['event_type'])
df_fast = pd.concat([df, event_dummies], axis=1)

session_df = df_fast.groupby('user_session').agg(
    user_id         = ('user_id', 'first'),
    session_start   = ('event_time', 'min'),
    session_end     = ('event_time', 'max'),
    total_events    = ('event_type', 'count'),
    num_views       = ('view', 'sum'),
    num_cart        = ('cart', 'sum'),
    num_purchase    = ('purchase', 'sum'),
    num_remove      = ('remove_from_cart', 'sum'),
    num_returns     = ('is_return', 'sum'),      
    unique_products = ('product_id', 'nunique'),
    avg_price       = ('price', 'mean'),
    max_price       = ('price', 'max'),
    hour            = ('hour', 'min'),
    day_of_week     = ('day_of_week', 'min'),
).reset_index()

print(session_df.shape)
print(session_df.head())

(4535940, 15)
                           user_session    user_id             session_start  \
0  0000061d-f3e9-484b-8c73-e54f355032a3  539262914 2020-01-16 06:30:41+03:00   
1  00000ac8-0015-4f12-996a-be2896323738  605114412 2020-01-25 01:22:20+03:00   
2  00000dd2-0f5d-4fc9-9d6b-2fc8c7514b04  556321594 2019-11-05 10:57:05+03:00   
3  000013d6-68a4-40cf-9452-6577dbfab515  405771061 2019-10-23 12:07:38+03:00   
4  00001aa1-7ee6-4d8c-81ff-bd5ce1dd1d6e  586931185 2019-12-20 23:37:29+03:00   

                session_end  total_events  num_views  num_cart  num_purchase  \
0 2020-01-16 06:30:41+03:00             1          1         0             0   
1 2020-01-25 01:22:20+03:00             1          1         0             0   
2 2019-11-05 10:57:05+03:00             1          1         0             0   
3 2019-10-23 14:12:21+03:00            20          2         1             9   
4 2019-12-20 23:50:05+03:00             2          2         0             0   

   num_remove  num_retur

In [ ]:
# add one more feature: session duration
session_df['session_duration_min'] = (
    (session_df['session_end'] - session_df['session_start'])
    .dt.total_seconds() / 60
)

print(session_df['session_duration_min'].describe())

count    4.535940e+06
mean     2.568981e+02
std      5.154252e+03
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.550000e+00
max      2.183259e+05
Name: session_duration_min, dtype: float64


In [64]:
print(session_df['session_duration_min'].quantile([0.95, 0.99, 0.999]))
print(f"Sessions > 1440min (24hr): {(session_df['session_duration_min'] > 1440).sum()}")
print(f"Sessions > 60min: {(session_df['session_duration_min'] > 60).sum()}")

0.950        41.916667
0.990       962.800833
0.999    102160.941167
Name: session_duration_min, dtype: float64
Sessions > 1440min (24hr): 36195
Sessions > 60min: 174563


In [ ]:
# Do not drop sessions; only set abnormal duration to None,now the duration looks logical
session_df.loc[session_df['session_duration_min'] > 1440, 'session_duration_min'] = None

print(f"Null durations: {session_df['session_duration_min'].isna().sum()}")
print(session_df['session_duration_min'].describe())

Null durations: 36195
count    4.499745e+06
mean     1.087187e+01
std      7.161071e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.416667e+00
max      1.439983e+03
Name: session_duration_min, dtype: float64


In [69]:
session_df.head(1)

,user_session,user_id,session_start,session_end,total_events,num_views,num_cart,num_purchase,num_remove,num_returns,unique_products,avg_price,max_price,hour,day_of_week,session_duration_min
0,0000061d-f3e9-484b-8c73-e54f355032a3,539262914,2020-01-16 06:30:41+03:00,2020-01-16 06:30:41+03:00,1,1,0,0,0,0,1,194.44,194.44,6,Thursday,0.0


In [ ]:
# create [converted]as the target for model building
session_df['converted'] = (session_df['num_purchase'] > 0).astype(int)

print(f"Total sessions: {len(session_df)}")
print(f"Converted sessions: {session_df['converted'].sum()}")
print(f"Conversion rate: {session_df['converted'].mean():.2%}")

Total sessions: 4535940
Converted sessions: 155617
Conversion rate: 3.43%


In [72]:
session_df.head(1)

,user_session,user_id,session_start,session_end,total_events,num_views,num_cart,num_purchase,num_remove,num_returns,unique_products,avg_price,max_price,hour,day_of_week,session_duration_min,converted
0,0000061d-f3e9-484b-8c73-e54f355032a3,539262914,2020-01-16 06:30:41+03:00,2020-01-16 06:30:41+03:00,1,1,0,0,0,0,1,194.44,194.44,6,Thursday,0.0,0


In [71]:
df.to_csv('events_cleaned.csv', index=False)
session_df.to_csv('sessions_cleaned.csv', index=False)